# BUSINESS CASE 2: TRANSPARENT FINANCIAL ADVISORY PIPELINE (ADVANCED EDITION)
**Author:** [Your Name/Group] | **Course:** [Course Name] | **Environment:** Google Colab / Local

### Executive Summary
This notebook implements a high-precision Machine Learning pipeline for routing clients toward financial investment products, ensuring full compliance with the **MiFID II** directive and the **European AI Act**.

The architecture is built on the **Transparent Glassbox** paradigm:
1. **Explainability by Design:** We utilize **Explainable Boosting Machines (EBM)** to guarantee the exact traceability and interpretability of every single feature. Unlike "Black Box" models (like deep neural networks or standard XGBoost), EBMs provide white-box transparency without relying on post-hoc approximations.
2. **Rigorous Anti-Leakage Framework:** A dual-stage data contract is implemented via a custom Transformer. This ensures that all statistical parameters (imputation, clipping, ratios) are calculated strictly within the training folds, preventing any look-ahead bias.
3. **Regulatory-Compliant Recommender:** Final recommendations are not based on raw probabilities alone but are passed through a compliance engine that enforces constraints on Risk Propensity, Age, Wealth, and Financial Literacy.



## 1. Environment Setup & Dependency Management
This section ensures that all necessary libraries are installed and configured. We specifically install `interpret` for Glassbox models and `optuna` for Bayesian hyperparameter optimization.



In [ ]:
import os
import sys
import subprocess
import json
import pickle
import warnings
import time
import shutil
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import (
    roc_auc_score, brier_score_loss,
    precision_score, recall_score, f1_score,
    precision_recall_curve, average_precision_score, roc_curve
)
from sklearn.utils import resample
import optuna

def install_dependencies():
    """Installs required packages if they are not already available in the environment."""
    required = {"interpret": "interpret", "optuna": "optuna", "xgboost": "xgboost", "plotly": "plotly", "openpyxl": "openpyxl", "xlrd": "xlrd"}
    missing = [pkg for mod, pkg in required.items() if not __import__('importlib.util').util.find_spec(mod)]
    if missing:
        print(f"📦 Installing missing dependencies: {', '.join(missing)}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + missing)
    print("✅ Core dependencies ready.")

install_dependencies()
from xgboost import XGBClassifier
from interpret.glassbox import ExplainableBoostingClassifier

# Global configuration
warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)
plt.style.use('seaborn-v0_8-muted')
sns.set_theme(style="whitegrid")

# Environment-aware pathing (Support for local and Google Colab)
try:
    from google.colab import drive
    COLAB_ENV = True
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/PipelineX'
except ImportError:
    COLAB_ENV = False
    PROJECT_ROOT = ".."

PIPELINE_X_DIR = os.path.join(PROJECT_ROOT, 'Production_Artifacts' if COLAB_ENV else 'Output/Pipeline_X')
RAW_DATA_PATH  = os.path.join(PROJECT_ROOT, 'Dataset2_Needs.xls')
os.makedirs(PIPELINE_X_DIR, exist_ok=True)

# Dataset Paths
BASE_DATA_PATH   = os.path.join(PIPELINE_X_DIR, "Dataset_Needs_Frozen.csv")
MASTER_DATA_PATH = os.path.join(PIPELINE_X_DIR, "Master_Needs_X.csv")

# Pipeline Constants
TARGET_COLS  = ["AccumulationInvestment", "IncomeInvestment"]
RANDOM_STATE = 42
FOLD_COL     = "stratified_fold"
N_TRIALS_XGB = 10  
N_TRIALS_EBM = 10  
THRESHOLD_ACC = 0.50
THRESHOLD_INC = 0.50



## 2. Architecture: The Anti-Leakage Transformer
At the heart of our data contract is the `PipelineXTransformer`. 

**Anti-Leakage Logic:**
- **Fit Step:** Only computes statistics (medians, quantiles, max values) from the training data.
- **Transform Step:** Applies these stored statistics to any future data (validation or test).

**Key Feature Engineering:**
- **Age Brackets:** Captures non-linear life-stage effects.
- **Wealth/Age Ratio:** A proxy for savings speed and wealth accumulation capacity over time.
- **Per-Person Stats:** Adjusts income and wealth based on family size to better capture disposable resources.



In [ ]:
class PipelineXTransformer:
    def __init__(self):
        self.medians, self.p99_inc, self.p99_wth, self.inc_max = None, None, None, None
        self.is_fitted = False

    def fit(self, df_train: pd.DataFrame):
        """Learns statistical distributions strictly from the training portion."""
        df = df_train.copy()
        # Outlier clipping thresholds (99th percentile)
        self.p99_inc = df["Income"].quantile(0.99)
        self.p99_wth = df["Wealth"].quantile(0.99)
        self.inc_max = df["Income"].max()
        # Medians for robust imputation
        self.medians = df.median(numeric_only=True)
        self.is_fitted = True
        return self

    def transform(self, df_in: pd.DataFrame) -> pd.DataFrame:
        """Applies learned statistics to transform features without look-ahead bias."""
        if not self.is_fitted: raise RuntimeError("Transformer must be fitted first.")
        df = df_in.copy()
        df.fillna(self.medians, inplace=True)
        
        # Life-Stage Encoding
        df["Age_bracket"] = pd.cut(df["Age"], bins=[17, 35, 55, 120], labels=["Young", "Mid", "Senior"])
        dummies = pd.get_dummies(df["Age_bracket"], prefix="Age_bracket", dtype=int)
        for label in ["Age_bracket_Young", "Age_bracket_Mid", "Age_bracket_Senior"]:
            if label not in dummies.columns: dummies[label] = 0
        df = pd.concat([df.drop(columns=["Age_bracket"]), dummies[["Age_bracket_Young", "Age_bracket_Mid", "Age_bracket_Senior"]]], axis=1)
        
        # Financial Ratio Engineering
        clipped_inc = df["Income"].clip(upper=self.p99_inc)
        clipped_wth = df["Wealth"].clip(upper=self.p99_wth)
        df["Wealth_log"] = np.log1p(df["Wealth"])
        df["Income_log"] = np.log1p(df["Income"])
        
        # Advanced Wealth Dynamics
        adult_years = (df["Age"] - 17).clip(lower=1)
        df["Wealth_Age_Ratio_log"] = np.log1p(clipped_wth / adult_years)
        
        # Per-Capita Indicators
        safe_fm = df["FamilyMembers"].replace(0, np.nan).fillna(self.medians.get("FamilyMembers", 1))
        df["Wealth_per_person"] = clipped_wth / safe_fm
        df["Income_per_person"] = clipped_inc / safe_fm
        
        # Income/Wealth Balance
        safe_wealth = clipped_wth.replace(0, np.nan)
        raw_ratio = clipped_inc.div(safe_wealth).fillna(self.inc_max)
        df["Income_Wealth_Ratio_log"] = np.log1p(raw_ratio)
        return df

# Data Access Utilities
_df_cache = {"base": None, "master": None}
def _get_df(stage):
    if _df_cache[stage] is not None: return _df_cache[stage]
    path = BASE_DATA_PATH if stage == "base" else MASTER_DATA_PATH
    if not os.path.exists(path): return None
    df = pd.read_csv(path)
    _df_cache[stage] = df
    return df

def get_feature_cols():
    if not os.path.exists(MASTER_DATA_PATH): raise FileNotFoundError("Feature engineering (Phase 01) must be run first.")
    cols = pd.read_csv(MASTER_DATA_PATH, nrows=0).columns.tolist()
    return [c for c in cols if c not in TARGET_COLS + ["ID", FOLD_COL] and not c.startswith("Unnamed")]

def get_full_train_val(stage="master"):
    df = _get_df(stage)
    if df is None: return None, None
    mask = df[FOLD_COL] >= 0
    feats = [c for c in df.columns if c not in TARGET_COLS + ["ID", FOLD_COL]]
    return df[mask][["ID"] + feats], df[mask][TARGET_COLS]

def get_test_set(stage="master"):
    df = _get_df(stage)
    if df is None: return None, None
    mask = df[FOLD_COL] == -1
    feats = [c for c in df.columns if c not in TARGET_COLS + ["ID", FOLD_COL]]
    return df[mask][["ID"] + feats], df[mask][TARGET_COLS]

def get_cv_splitter(stage="master"):
    df = _get_df(stage)
    if df is None: return []
    tv_df = df[df[FOLD_COL] >= 0].reset_index(drop=True)
    return [(tv_df.index[tv_df[FOLD_COL] != f].tolist(), tv_df.index[tv_df[FOLD_COL] == f].tolist()) for f in range(5)]



## 3. Phase 00: Dataset Freezing & Stratification
We create an immutable "Frozen" version of the dataset. 

**Key Logic:**
- We stratify on a **combined target** (concatenation of both investment needs) to ensure that both the training and test sets have an identical distribution of target classes.
- This phase ensures that all future experiments (Phase 01-07) are conducted on the exact same indices for fair comparison.



In [ ]:
print("[00x] Freezing dataset with stratified folds...")
df_work = pd.read_excel(RAW_DATA_PATH, sheet_name="Needs")
df_work.columns = df_work.columns.str.strip()

# Create a combined stratification key
df_work["stratify_combined"] = df_work["AccumulationInvestment"].astype(str) + "_" + df_work["IncomeInvestment"].astype(str)

indices = np.arange(len(df_work))
train_idx, test_idx = train_test_split(indices, test_size=1000, stratify=df_work["stratify_combined"], random_state=RANDOM_STATE)

df_work[FOLD_COL] = -5 
df_work.iloc[test_idx, df_work.columns.get_loc(FOLD_COL)] = -1

# Split training set into 5 folds for cross-validation
df_train_only = df_work.iloc[train_idx].copy()
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

for f_id, (_, val_rel_idx) in enumerate(skf.split(np.zeros(len(df_train_only)), df_train_only["stratify_combined"])):
    abs_idx = train_idx[val_rel_idx]
    df_work.iloc[abs_idx, df_work.columns.get_loc(FOLD_COL)] = f_id

df_work.drop(columns=["stratify_combined"], inplace=True)
df_work.to_csv(BASE_DATA_PATH, index=False)
print(f"✅ Frozen Dataset created at: {BASE_DATA_PATH}")



## 4. Phase 01: Master Feature Engineering
This section applies the `PipelineXTransformer` to the frozen dataset. Note that the statistics are learned only once from the global training set (80% of data) and applied to everything. This simulates a production environment where the transformer is saved as a persistent artifact.



In [ ]:
print("[01x] Processing features and saving transformer...")
df_base = pd.read_csv(BASE_DATA_PATH)
df_tr_ref = df_base[df_base[FOLD_COL] >= 0]
transformer = PipelineXTransformer().fit(df_tr_ref)

# Persist transformer for API inference deployment
with open(os.path.join(PIPELINE_X_DIR, "production_transformer.pkl"), "wb") as f:
    pickle.dump(transformer, f)

df_master = transformer.transform(df_base)
df_master.to_csv(MASTER_DATA_PATH, index=False)
_df_cache["master"] = None # Invalidate cache to force reload
print("✅ Master Dataset generated. Feature extraction complete.")



## 5. Phase 02: Calibrated XGBoost Baseline
We establish a performance baseline using **XGBoost**.

**Refined Training Procedure:**
- **Bayesian Tuning:** Optuna optimizes 9 separate hyperparameters.
- **State Isolation:** In every cross-validation trial, a new model instance is created per fold to prevent state carry-over.
- **Isotonic Calibration:** We use `ensemble=False`. This ensures that after finding the best parameters, a single model is fitted on the full training set (4000 rows), providing a smooth and granular isotonic curve for optimal test set ranking.



In [ ]:
print("[02x] Optimizing Calibrated XGBoost...")
X_tv_base, y_tv_base = get_full_train_val(stage="base")
cv_splits = get_cv_splitter(stage="base")
FEATURE_COLS = get_feature_cols()

def make_xgb_objective(target_name):
    def objective(trial):
        start_time = time.time()
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 200, 800, step=100),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
            "max_depth": trial.suggest_int("max_depth", 3, 8),
            "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
            "subsample": trial.suggest_float("subsample", 0.6, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
            "eval_metric": "logloss",
            "tree_method": "hist",
            "random_state": RANDOM_STATE,
            "n_jobs": -1
        }
        fold_aucs = []
        for i, (tr_idx, val_idx) in enumerate(cv_splits):
            model = XGBClassifier(**params)
            X_tr_raw, X_va_raw = X_tv_base.iloc[tr_idx], X_tv_base.iloc[val_idx]
            y_tr_f, y_va_f = y_tv_base[target_name].iloc[tr_idx].values, y_tv_base[target_name].iloc[val_idx].values
            
            # Anti-leakage transform per fold
            trans = PipelineXTransformer().fit(X_tr_raw)
            X_tr_f = trans.transform(X_tr_raw)[FEATURE_COLS].values
            X_va_f = trans.transform(X_va_raw)[FEATURE_COLS].values
            
            model.fit(X_tr_f, y_tr_f)
            fold_aucs.append(roc_auc_score(y_va_f, model.predict_proba(X_va_f)[:, 1]))
            
        return np.mean(fold_aucs)
    return objective

# Execution of tuning and final fit
performance_xgb = []
X_tv_ms, y_tv_ms = get_full_train_val(stage="master")
X_test_ms, y_test_ms = get_test_set(stage="master")

for target in TARGET_COLS:
    print(f"  -> Tuning XGBoost for {target}...")
    study = optuna.create_study(direction="maximize")
    study.optimize(make_xgb_objective(target), n_trials=N_TRIALS_XGB)
    
    # Final production fit with ensemble=False
    final_params = {"eval_metric": "logloss", "tree_method": "hist", "random_state": RANDOM_STATE, "n_jobs": -1, **study.best_params}
    base_model = XGBClassifier(**final_params)
    calibrated = CalibratedClassifierCV(base_model, method="isotonic", cv=cv_splits, ensemble=False)
    calibrated.fit(X_tv_ms[FEATURE_COLS].values, y_tv_ms[target].values)
    
    with open(os.path.join(PIPELINE_X_DIR, f"xgb_{target[:3].lower()}_calibrated.pkl"), "wb") as f:
        pickle.dump(calibrated, f)
        
    probs = calibrated.predict_proba(X_test_ms[FEATURE_COLS].values)[:, 1]
    auc = roc_auc_score(y_test_ms[target].values, probs)
    performance_xgb.append({"Target": target, "XGB_Test_AUC": auc})

print("✅ XGBoost Evaluation Results:")
display(pd.DataFrame(performance_xgb))



## 6. Phase 03: The Glassbox Champion (EBM)
This phase implements the **Explainable Boosting Machine (EBM)**. EBMs are Generalized Additive Models (GAMs) that operate like boosted trees but produce white-box models that disclose the exact contribution of each feature.

**Strategic Decision:**
- **No Calibration:** We avoid post-fit calibration on EBMs to preserve their inherent interpretability (the raw scores directly map to feature contributions).
- **High Granularity:** We enable `outer_bags=8` to stabilize the feature interaction effects.



In [ ]:
print("[03x] Training Glassbox EBM Models...")
# Exclude dummy features from EBM to let it discover age trends natively
ebm_f = [c for c in FEATURE_COLS if not c.startswith("Age_bracket")]
ebm_raw_models = {}
performance_ebm = []

def make_ebm_objective(target_name):
    def objective(trial):
        params = {
            "feature_names": ebm_f,
            "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.05, log=True),
            "max_bins": trial.suggest_categorical("max_bins", [128, 256, 512]),
            "interactions": trial.suggest_int("interactions", 5, 20),
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 2, 12),
            "outer_bags": 14, "inner_bags": 0, "random_state": RANDOM_STATE, "n_jobs": -1
        }
        fold_aucs = []
        for i, (tr_idx, val_idx) in enumerate(cv_splits):
            model = ExplainableBoostingClassifier(**params)
            X_tr_raw = X_tv_base.iloc[tr_idx]
            X_va_raw = X_tv_base.iloc[val_idx]
            y_tr_f = y_tv_base[target_name].iloc[tr_idx].values
            y_va_f = y_tv_base[target_name].iloc[val_idx].values
            
            trans = PipelineXTransformer().fit(X_tr_raw)
            model.fit(trans.transform(X_tr_raw)[ebm_f].values, y_tr_f)
            fold_aucs.append(roc_auc_score(y_va_f, model.predict_proba(trans.transform(X_va_raw)[ebm_f].values)[:, 1]))
            
        return np.mean(fold_aucs)
    return objective

for target in TARGET_COLS:
    print(f"  -> Tuning Glassbox EBM for {target}...")
    study = optuna.create_study(direction="maximize")
    study.optimize(make_ebm_objective(target), n_trials=N_TRIALS_EBM)
    
    # Final raw EBM fit (Interpretation focus)
    final_ebm_params = {"feature_names": ebm_f, "outer_bags": 14, "inner_bags": 0, "random_state": RANDOM_STATE, "n_jobs": -1, **study.best_params}
    model_raw = ExplainableBoostingClassifier(**final_ebm_params)
    model_raw.fit(X_tv_ms[ebm_f].values, y_tv_ms[target].values)
    
    ebm_raw_models[target] = model_raw
    with open(os.path.join(PIPELINE_X_DIR, f"ebm_{target[:3].lower()}_model.pkl"), "wb") as f:
        pickle.dump(model_raw, f)
        
    probs = model_raw.predict_proba(X_test_ms[ebm_f].values)[:, 1]
    test_auc = roc_auc_score(y_test_ms[target].values, probs)
    performance_ebm.append({"Target": target, "EBM_Test_AUC": test_auc})

print("✅ EBM Glassbox Evaluation (Production Model):")
display(pd.DataFrame(performance_ebm))



## 7. Global Feature Importance (Glassbox Audit)
One of the key benefits of EBM is the ability to show the relative importance of each feature across the entire dataset. This is essential for auditing model fairness and regulatory compliance.



In [ ]:
for target in TARGET_COLS:
    ebm = ebm_raw_models[target]
    print(f"\n--- {target}: Global Feature Importance ---")
    ebm_global = ebm.explain_global()
    
    # Extract data for visualization
    names = ebm_global.data()['names']
    scores = ebm_global.data()['scores']
    
    plt.figure(figsize=(10, 6))
    indices = np.argsort(scores)
    plt.barh([names[i] for i in indices], [scores[i] for i in indices], color='teal', alpha=0.8)
    plt.title(f"EBM Global Feature Importance - {target}", fontweight='bold')
    plt.xlabel("Absolute Score (Logit Contribution)")
    plt.tight_layout()
    plt.show()



## 8. Phase 05: Product Recommendation & Regulatory Engine
We translate model probabilities into actual investment recommendations.

**Advising Logic (MiFID II Alignment):**
1. **Target Identification:** Determine if the client needs Accumulation, Income, or both.
2. **Risk Filtering:** Ensure the product risk does not exceed the client's `RiskPropensity`.
3. **Life-Stage Guardrails:** Elderly clients (>75) are restricted from high-risk growth products (Products 2, 6, 7, 8).
4. **Wealth Guardrails:** Wealth-intensive products (9, 10, 11) require a minimum wealth of 100k.
5. **Financial Literacy:** Complex products are restricted for clients with low education scores (< 0.3).



In [ ]:
print("[05x] Executing Compliance-Driven Recommender...")
products_df = pd.read_excel(RAW_DATA_PATH, sheet_name="Products")

def match_advisory(profile, p_df):
    """Business logic for MiFID II compliant product matching."""
    n, r, a, w, e = profile["need"], profile["risk"], profile["age"], profile["wth"], profile["edu"]
    if n == "None": return None
    
    type_filter = 1 if n in ["Accumulation", "Both"] else 0
    filtered = p_df[(p_df["Type"] == type_filter) & (p_df["Risk"] <= r)].copy()
    
    # Compliance Overlays
    if a > 75:     filtered = filtered[~filtered["IDProduct"].isin([2, 6, 7, 8])] 
    if w < 100000: filtered = filtered[~filtered["IDProduct"].isin([9, 10, 11])]
    if e < 0.3:    filtered = filtered[~filtered["IDProduct"].isin([7, 11])]
    
    # Priority for hybrid needs
    if n == "Both" and not filtered[filtered["IDProduct"].isin([4, 8])].empty:
        filtered = filtered[filtered["IDProduct"].isin([4, 8])]

    return filtered.sort_values("Risk", ascending=False).iloc[0]["IDProduct"] if not filtered.empty else None

# Simulation of API inference using the Production Transformer and EBM models
X_te_base, y_te_df = get_test_set(stage="base")
with open(os.path.join(PIPELINE_X_DIR, "production_transformer.pkl"), "rb") as f:
    prod_trans = pickle.load(f)

X_te_processed = prod_trans.transform(X_te_base)
p_acc = ebm_raw_models["AccumulationInvestment"].predict_proba(X_te_processed[ebm_f].values)[:, 1]
p_inc = ebm_raw_models["IncomeInvestment"].predict_proba(X_te_processed[ebm_f].values)[:, 1]

recommendations = []
for i in range(len(X_te_processed)):
    row = X_te_processed.iloc[i]
    is_acc, is_inc = p_acc[i] >= THRESHOLD_ACC, p_inc[i] >= THRESHOLD_INC
    need_label = "Both" if is_acc and is_inc else ("Accumulation" if is_acc else ("Income" if is_inc else "None"))
    
    client_profile = {
        "need": need_label, "risk": row["RiskPropensity"], "age": row["Age"], 
        "wth": row["Wealth"], "edu": row["FinancialEducation"]
    }
    
    recommendations.append({
        "ID": row["ID"], "Gender": row["Gender"], "Age": row["Age"], "Wealth": row["Wealth"],
        "Need": need_label, "Recommended_Product": match_advisory(client_profile, products_df),
        "Score_Acc": p_acc[i], "Score_Inc": p_inc[i]
    })

results_df = pd.DataFrame(recommendations)
results_df.to_csv(os.path.join(PIPELINE_X_DIR, "final_advisory_results.csv"), index=False)
print("✅ Recommendations localized. Sample results:")
display(results_df.head(5))



## 9. Phase 06: Compliance & Fairness Audit
We audit the results to ensure no demographic bias (Gender Parity). The AI Act requires that automated financial decisions do not manifest discrimination.



In [ ]:
results_df["Approved"] = results_df["Recommended_Product"].notna()
fairness = results_df.groupby("Gender")["Approved"].mean()

plt.figure(figsize=(8, 5))
sns.barplot(x=["Male (0)", "Female (1)"], y=fairness.values, palette="mako")
plt.title("Fairness Audit: Product Approval Rate by Gender", fontweight="bold")
plt.axhline(0.8, color="red", linestyle="--", label="Regulatory Min Threshold (80%)")
plt.ylabel("Approval Rate")
plt.legend()
plt.show()



## 10. Phase 07: Strategic Business Dashboard
We visualize the strategic allocation of clients across different segments (Age vs Wealth).



In [ ]:
results_df["Age_Bin"] = pd.cut(results_df["Age"], bins=[17, 35, 55, 120], labels=["18-35", "36-55", "56+"])
results_df["Wealth_Bin"] = pd.qcut(results_df["Wealth"], q=4, labels=["Q1", "Q2", "Q3", "Q4"])
pivot_map = results_df[results_df["Approved"]].pivot_table(index="Age_Bin", columns="Wealth_Bin", values="ID", aggfunc="count", fill_value=0)

plt.figure(figsize=(10, 6))
sns.heatmap(pivot_map, annot=True, fmt="d", cmap="YlGnBu", cbar_kws={'label': 'Client Count'})
plt.title("Strategic Allocation Map: Wealth vs Age Segments", fontweight="bold")
plt.tight_layout()
plt.show()



## 11. Final Metric: ROI Lift Curve
The Lift Curve measures the commercial improvement compared to a random selection strategy. It shows how many "ideal" targets we capture by prioritizing top-model scores.



In [ ]:
results_df["Max_Score"] = results_df[["Score_Acc", "Score_Inc"]].max(axis=1)
results_df.sort_values("Max_Score", ascending=False, inplace=True)
results_df["Cumulative_Targets"] = (results_df["Max_Score"] >= 0.5).astype(int).cumsum() / (results_df["Max_Score"] >= 0.5).sum()
results_df["Client_Percentage"] = (np.arange(len(results_df)) + 1) / len(results_df)

plt.figure(figsize=(10, 6))
plt.plot(results_df["Client_Percentage"], results_df["Cumulative_Targets"], color="#3498DB", lw=4, label="Glassbox Pipeline")
plt.plot([0, 1], [0, 1], "k--", alpha=0.4, label="Random Benchmark")
plt.fill_between(results_df["Client_Percentage"], results_df["Client_Percentage"], results_df["Cumulative_Targets"], color="#3498DB", alpha=0.1)
plt.title("Commercial Performance: Lift Curve", fontweight="bold")
plt.xlabel("Percentage of Clients Contacted")
plt.ylabel("Cumulative Proportion of Needs Identified")
plt.legend()
plt.tight_layout()
plt.show()

print("\n🚀 Pipeline Transparent Execution Finished. All artifacts are saved in the Output directory.")
